## Fetch Company Tickers

- Our goal currently is to automatically extract company ticker information from the a trusted source i.e [CNBC Dow30](https://www.cnbc.com/dow-30/).
- We devise this action currently by using relevant libraries for extraction from the above seed URL.
- Later after getting the relevant links, we reorganize this data and aggregate the data across three/four properties -- Company Name, Ticker, Homepage URL.
- We validate the output before going to the next phase -- fetching IR URl.

Current Issues

- We are not able to extract information from the website for the primary reason that majority of the websites operate on a Javascript loading basis. So we need a library that simulates that...

> Using playwright as our primary library for broswer based extraction. 

In [ ]:
from playwright.async_api import async_playwright

async def scrape_apple_ir():
    """Scrape Apple Investor Relations page"""
    
    # Create the playwright instance
    async with async_playwright() as p:
        # Launch browser inside the context
        browser = await p.chromium.launch(headless=True)
        
        # Create page inside browser context
        page = await browser.new_page()
        
        try:
            print("🌐 Loading Apple Investor Relations...")
            
            # Navigate to page
            await page.goto('https://investor.apple.com/', wait_until='networkidle', timeout=30000)
            
            print("✓ Page loaded!")
            
            # Wait a bit for dynamic content
            await page.wait_for_timeout(2000)
            
            # Get page title to confirm it loaded
            title = await page.title()
            print(f"📄 Page title: {title}")
            
            # Get all PDF links
            pdf_links = await page.locator('a[href*=".pdf"]').all()
            print(f"📑 Found {len(pdf_links)} PDF links")
            
            # Extract link data
            results = []
            for link in pdf_links[:10]:  # First 10 links
                try:
                    href = await link.get_attribute('href')
                    text = await link.inner_text()
                    results.append({
                        'text': text.strip(),
                        'url': href
                    })
                except:
                    continue
            
            print(f"✓ Extracted {len(results)} links")
            
            return results
            
        except Exception as e:
            print(f"❌ Error: {e}")
            return []
            
        finally:
            # Always close browser
            await browser.close()

# Run the function
results = await scrape_apple_ir()

# Display results
print("\n📋 Results:")
for i, r in enumerate(results, 1):
    print(f"{i}. {r['text'][:60]} -> {r['url']}")

🌐 Loading Apple Investor Relations...
✓ Page loaded!
📄 Page title: Investor Relations - Apple
📑 Found 37 PDF links
✓ Extracted 10 links

📋 Results:
1. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2025-q3/FY25_Q3_Consolidated_Financial_Statements.pdf
2. 10-Q -> https://s2.q4cdn.com/470004039/files/doc_earnings/2025/q3/filing/10Q-Q3-2025-as-filed.pdf
3. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2025-q2/FY25_Q2_Consolidated_Financial_Statements.pdf
4. 10-Q -> https://s2.q4cdn.com/470004039/files/doc_earnings/2025/q2/filing/10Q-Q2-2025-as-filed.pdf
5. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2025-q1/FY25_Q1_Consolidated_Financial_Statements.pdf
6. 10-Q -> https://s2.q4cdn.com/470004039/files/doc_earnings/2025/q1/filing/10Q-Q1-2025-as-filed.pdf
7. Financial Statements -> https://www.apple.com/newsroom/pdfs/fy2024-q4/FY24_Q4_Consolidated_Financial_Statements.pdf
8. 10-K -> https://s2.q4cdn.com/470004039/files/doc_earnings/2024/q4/fili

### Extraction of Tickers and Link Information from CNBC DOW30 Webpage

In [19]:
CNBC_DOW30_URL='https://www.cnbc.com/dow-30/'

In [ ]:
# An algorithm to determine high confidence links to lead to the IR page of the company after potentially multiple clicks, input of this algorithm being the whole webpage (URL)

async def find_potential_ir_paths(page, company_name, ticker, max_depth=3, visited=None):
    """
    Find potential paths that could lead to IR pages through multiple clicks.
    Uses breadth-first search with heuristics to explore promising paths.
    
    Args:
        page: Playwright page object
        company_name: Name of the company
        ticker: Stock ticker symbol
        max_depth: Maximum number of links to follow
        visited: Set of already visited URLs
        
    Returns:
        List of potential paths to IR pages, with confidence scores
    """
    if visited is None:
        visited = set()
        
    ir_keywords = ['investor', 'investors', 'ir', 'shareholders', 'financials', 'stock']
    menu_keywords = ['about', 'company', 'corporate', 'menu', 'navigation']
    
    async def score_link(link):
        try:
            href = await link.get_attribute('href')
            text = await link.inner_text()
            if not href or not text:
                return None, 0
                
            score = 0
            href_lower = href.lower()
            text_lower = text.lower()
            
            # Direct IR indicators
            for kw in ir_keywords:
                if kw in href_lower:
                    score += 3
                if kw in text_lower:
                    score += 2
                    
            # Navigation/menu indicators that might lead to IR
            for kw in menu_keywords:
                if kw in href_lower or kw in text_lower:
                    score += 1
                    
            # Company identifiers presence
            if company_name.lower() in href_lower or company_name.lower() in text_lower:
                score += 1
            
            # Ticker presence (both uppercase and lowercase)
            if ticker.upper() in href or ticker.upper() in text:
                score += 2
            if ticker.lower() in href_lower or ticker.lower() in text_lower:
                score += 2
                
            # Stock-specific patterns
            stock_patterns = [
                f"stock/{ticker}", 
                f"quote/{ticker}",
                "nyse:",
                "nasdaq:",
                f"{ticker.lower()}-stock",
                f"{ticker.upper()}-stock"
            ]
            for pattern in stock_patterns:
                if pattern in href_lower:
                    score += 2
                    
            return {'url': href, 'text': text}, score
            
        except:
            return None, 0
            
    async def explore_page(current_page, depth=0, path=None):
        if path is None:
            path = []
            
        if depth >= max_depth:
            return []
            
        current_url = current_page.url
        if current_url in visited:
            return []
            
        visited.add(current_url)
        
        links = await current_page.locator('a').all()
        paths = []
        
        # Score and sort links on current page
        scored_links = []
        for link in links:
            link_info, score = await score_link(link)
            if link_info and score > 0:
                scored_links.append((link_info, score))
                
        scored_links.sort(key=lambda x: x[1], reverse=True)
        
        # Explore most promising links
        for link_info, score in scored_links[:3]:  # Limit branching factor
            new_path = path + [link_info]
            
            # If high confidence IR page, add path
            if score >= 4:
                paths.append({
                    'path': new_path,
                    'confidence': score
                })
                
            # Otherwise explore further if score is promising
            elif score >= 2:
                try:
                    new_page = await current_page.context.new_page()
                    await new_page.goto(link_info['url'])
                    sub_paths = await explore_page(new_page, depth + 1, new_path)
                    paths.extend(sub_paths)
                    await new_page.close()
                except:
                    continue
                    
        return paths
        
    paths = await explore_page(page)
    paths.sort(key=lambda x: x['confidence'], reverse=True)
    
    return paths


In [ ]:
# An algorithm to determine whether it is worth clicking on  a certain link to go to the final goal of getting an IR page/link?


In [ ]:
# An algorithm to determine whether a URL is the homepage of the company?

In [42]:
# An standard algorithm to determine whether clicking on a certain URL is worth it?
def evaluate_link_value(link_text, link_url, intent):
    """
    Evaluates whether a link is worth clicking to eventually reach the target page.
    Returns a score indicating the likelihood that following this link will lead to the target.
    
    Parameters:
    - link_text: The visible text of the link
    - link_url: The URL the link points to
    - intent: The intent of the user (to go to the IR page of a company, to go to the homepage of that company, etc.)
    Returns:
    - score: Float between 0-5 indicating value of clicking link
    """
    score = 0
    link_text = link_text.lower()
    link_url = link_url.lower()
    # intent expansion
    target_company = intent['target_company']
    target_type = intent['target_type']
    company_name = target_company['company_name'].lower()
    company_ticker = target_company['ticker'].lower()
    # Direct matches for IR/investor pages
    ir_terms = ['investor', 'ir/', 'investors', 'investor-relations', 'stockholder']
    if any(term in link_url or term in link_text for term in ir_terms):
        score += 2
        # Even better if company name/ticker also present
        if company_name in link_url or company_ticker in link_url:
            score += 2
            
    # Company identifier matches
    if company_name in link_url or company_ticker in link_url:
        score += 1
        
    # Navigation elements that could lead to IR
    nav_terms = ['about', 'corporate', 'company', 'finance']
    if any(term in link_url or term in link_text for term in nav_terms):
        score += 0.5
        
    # Penalize likely irrelevant pages
    bad_terms = ['products', 'shop', 'cart', 'login', 'careers', 'news']
    if any(term in link_url or term in link_text for term in bad_terms):
        score -= 1
        
    return max(0, score)  # Don't return negative scores


In [38]:
def sanitize_href(href,domain_name):
    # Check if the link is a relative URL
    if not href.startswith(('http://', 'https://')):
        # check if the href has a domain name
        if domain_name in href:
            # just add the https:// to the href
            href = f"https:{href}"
        else:
            # add the domain name to the href
            href = f"https://{domain_name}/{href}"
    return href

In [46]:
#Use playwright to extract the tickers and link information from the CNBC DOW30 Webpage
async def scrape_dow30_tickers():
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        try:
            page = await browser.new_page()
            await page.goto(CNBC_DOW30_URL)
            domain_name = page.url.split('/')[2]
            print("📱 Accessing CNBC DOW30 page...")
            print("Domain Name: ",domain_name)

            # Wait for the table to load
            await page.wait_for_selector('table')
            
            # Extract ticker information
            ticker_elements = await page.locator('tr').all()
            
            results = []
            for element in ticker_elements[1:]:  # Skip header row
                try:
                    # Get cells and links from each row
                    cells = await element.locator('td').all()
                    
                    # Get any links in the row
                    links = await element.locator('a').all()
                    # Normalize the links array to be containing absolute URLs
                    link_urls = []
                    for link in links:
                        href = await link.get_attribute('href')
                        # Normalize the link to be absolute URL
                        if href:
                            href = sanitize_href(href,domain_name)
                            # get the link text

                            # evaluate the link value
                            link_urls.append(href)
                    
                    if len(cells) >= 2:
                        symbol = await cells[0].inner_text()
                        company = await cells[1].inner_text()
                        for link_url in link_urls:
                            link_text = await link.inner_text()
                            intent = {
                                'target_company': {
                                    'company_name': company.strip(),
                                    'ticker': symbol.strip()
                                },
                                'target_type': 'IR'
                            }
                            link_value = evaluate_link_value(link_text, link_url, intent)
                            results.append({
                                'ticker': symbol.strip(),
                                'company_name': company.strip(),
                                'link_data': {
                                    'link_text': link_text,
                                    'link_url': link_url,
                                    'link_value': link_value
                                }
                            })
                except Exception as e:
                    print(f"Error processing row: {e}")
                    continue
                    
            print(f"✓ Extracted {len(results)} tickers")
            return results

        except Exception as e:
            print(f"❌ Error: {e}")
            return []
            
        finally:
            await browser.close()

# Run the function
dow30_tickers = await scrape_dow30_tickers()

# Display results
print("\n📋 DOW 30 Tickers:")
async with async_playwright() as p:
    browser = await p.chromium.launch()
    page = await browser.new_page()
    for ticker in dow30_tickers:
        # If the link value is greater than 0, then print the ticker, company name, link url and link value
        if ticker['link_data']['link_value'] > 0:
            print(f"{ticker['company_name']}: {ticker['link_data']['link_url']} {ticker['link_data']['link_value']}")
            # click on the link                
            await page.goto(ticker['link_data']['link_url'])
            # wait for the page to load
            # get the page title
            title = await page.title()
            print(f"Page Title: {title}")
    browser.close()

📱 Accessing CNBC DOW30 page...
Domain Name:  www.cnbc.com
✓ Extracted 30 tickers

📋 DOW 30 Tickers:
Goldman Sachs Group Inc: https://www.cnbc.com/quotes/GS 1
Page Title: GS: Goldman Sachs Group Inc - Stock Price, Quote and News - CNBC
Home Depot Inc: https://www.cnbc.com/quotes/HD 1
Page Title: HD: Home Depot Inc - Stock Price, Quote and News - CNBC
International Business Machines Corp: https://www.cnbc.com/quotes/IBM 1
Page Title: IBM: International Business Machines Corp - Stock Price, Quote and News - CNBC
Johnson & Johnson: https://www.cnbc.com/quotes/JNJ 1
Page Title: JNJ: Johnson & Johnson - Stock Price, Quote and News - CNBC
JPMorgan Chase & Co: https://www.cnbc.com/quotes/JPM 1
Page Title: JPM: JPMorgan Chase & Co - Stock Price, Quote and News - CNBC
McDonald’s Corp: https://www.cnbc.com/quotes/MCD 1
Page Title: MCD: McDonald's Corp - Stock Price, Quote and News - CNBC
3M Co: https://www.cnbc.com/quotes/MMM 1
Page Title: MMM: 3M Co - Stock Price, Quote and News - CNBC
Merck & C

/var/folders/9c/gldj5rjn0699dnx6qgv9jyqr0000gn/T/ipykernel_63746/3549897164.py:92: RuntimeWarning: coroutine 'Browser.close' was never awaited
  browser.close()
